# Onboarding Workflow Analysis

This notebook analyzes reconstructed and anonymized data for the Remote ERP Onboarding Workflow Case Study.

The notebook does not use raw company files, customer documents, credentials, or personal information.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Works from either repository root or notebooks/ directory
DATA_DIR = Path('../data')
if not DATA_DIR.exists():
    DATA_DIR = Path('data')

events = pd.read_csv(DATA_DIR / 'synthetic_onboarding_event_log.csv')
artifacts = pd.read_csv(DATA_DIR / 'artifact_inventory.csv')
taxonomy = pd.read_csv(DATA_DIR / 'onboarding_failure_taxonomy.csv')
roles = pd.read_csv(DATA_DIR / 'role_expectation_matrix.csv')
gaps = pd.read_csv(DATA_DIR / 'knowledge_transfer_gap_matrix.csv')

events['event_date'] = pd.to_datetime(events['event_date'], errors='coerce')
events.head()

## 1. Dataset overview

The first check counts reconstructed events, artifacts, taxonomy entries, role modes, and knowledge-transfer gap entries.

In [ ]:
overview = pd.Series({
    'reconstructed_events': len(events),
    'artifact_inventory_rows': len(artifacts),
    'failure_taxonomy_entries': len(taxonomy),
    'role_expectation_entries': len(roles),
    'knowledge_gap_entries': len(gaps),
})
overview

## 2. Events by onboarding phase

This shows which period contains more reconstructed onboarding events.

In [ ]:
phase_order = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'August/September']
monthly = events['phase_month'].value_counts().rename_axis('phase_month').reset_index(name='event_count')
monthly['order'] = monthly['phase_month'].apply(lambda x: phase_order.index(x) if x in phase_order else 999)
monthly = monthly.sort_values('order').drop(columns='order')
monthly

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(monthly['phase_month'], monthly['event_count'])
plt.title('Onboarding Events by Phase')
plt.xlabel('Phase')
plt.ylabel('Event count')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()

## 3. Risk-tag analysis

The risk tags show recurring CWI signals such as live project exposure, role ambiguity, artifact fragmentation, domain overload, and self-onboarding burden.

In [ ]:
risk_rows = []
for _, row in events.iterrows():
    tags = [tag.strip() for tag in str(row['onboarding_risk_tags']).split(';') if tag.strip()]
    for tag in tags:
        risk_rows.append({
            'phase_month': row['phase_month'],
            'domain': row['domain'],
            'role_mode': row['role_mode'],
            'risk_tag': tag,
        })

risk_tags = pd.DataFrame(risk_rows)
risk_counts = risk_tags['risk_tag'].value_counts().rename_axis('risk_tag').reset_index(name='count')
risk_counts.head(15)

In [ ]:
top_risks = risk_counts.head(12)
plt.figure(figsize=(10, 5))
plt.bar(top_risks['risk_tag'], top_risks['count'])
plt.title('Top Onboarding Risk Tags')
plt.xlabel('Risk tag')
plt.ylabel('Count')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 4. Domain and role-mode distribution

This section checks whether the onboarding environment was concentrated or distributed across many domains and role modes.

In [ ]:
domain_counts = events['domain'].value_counts().rename_axis('domain').reset_index(name='event_count')
role_mode_counts = events['role_mode'].value_counts().rename_axis('role_mode').reset_index(name='event_count')
domain_counts.head(12)

In [ ]:
role_mode_counts.head(12)

## 5. Failure taxonomy and knowledge-transfer gaps

The taxonomy and gap matrix convert the onboarding experience into reusable analytical categories.

In [ ]:
taxonomy[['failure_code', 'failure_type', 'severity', 'public_wording']]

In [ ]:
gaps[['knowledge_area', 'gap_type', 'recommended_scaffold', 'possible_cwi_metric']]

## 6. Interpretation

The reconstructed data suggests that the case should not be read merely as a difficult training experience. It shows a remote onboarding environment where live project artifacts, formal training, manufacturing ERP complexity, data reconciliation, presentation work, translation work, and self-study were compressed together.

From a CWI perspective, the central issue is the alignment failure among learning stage, artifact status, workflow sequence, role expectation, and feedback loop.